# N100 · 网格DP与路径状态

**目标：** 根据允许方向建立无环转移并处理障碍。

**先修：** N018, N098, N031。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 坐标状态；边界；路径计数；最小代价；逆向需求状态。

**配套讲解来源：** [同名逐章意见](../../comment/100_grid_dp.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

想象一个小人在 m 行 n 列的网格里，从左上角出发，每一步只能向右或向下走，要走到右下角。围绕这个场景，本章解决四个问题：

1. **有多少条不同的路径？** 比如 `unique_paths(3,7)` 返回 `28`：3 行 7 列的网格，从左上到右下共需 2 次向下、6 次向右，这些动作的排列方式共有 C(8,2)=28 种。
2. **网格里有障碍（1 表示障碍、0 表示空地）时还有多少路径？** 比如 `[[0,0],[0,1]]` 输出 `0`，因为右下角本身就是障碍，一条合法路径都没有。
3. **每格有一个花费，路径上花费总和最少是多少？** 比如网格 `[[1,3,1],[1,5,1]]` 输出 `6`，最优走法是"右、右、下"：1+3+1+1=6；走中间的 5 那格都要花更多。
4. **地牢游戏：每格会加血或扣血，血量任何时候都不能低于 1，出发前至少要带多少血？** 比如地牢 `[[-5]]`（进门就扣 5 点血）输出 `6`：进门前得有 6 点血，扣完 5 点还剩 1 点，恰好活着。

前三个问题都是从左上往右下"正向"推；第四个问题很反直觉——它必须**从右下角往左上角倒着推**，这是本章最重要的思维转弯。

## 2. 基础知识：先读懂这些词

- **网格 DP（坐标状态）**：状态就是"我现在站在哪个格子 (r, c)"，每个格子的答案由它依赖的邻居格子算出来。
- **DAG（有向无环图）依赖方向**：如果只能向右、向下走，那么任何格子的答案只依赖它上方和左方的格子，依赖关系不会绕圈。这保证了我们按从上到下、从左到右的顺序计算时，用到的值都早已算好。"无环"是 DP 能顺推的前提。
- **路径计数与最值转移**：数路径时，到达某格的路径数 = 上方来的路径数 + 左方来的路径数（相加）；求最小花费时，到达某格的最小花费 = min(从上方来的最小花费, 从左方来的最小花费) + 本格花费（取 min）。同一个框架，只是"合并方式"从加号换成 min。
- **滚动数组（压缩到一行）**：算第 r 行时只需要第 r−1 行的旧值，所以不必存整张表，一个长度为 n 的一维数组逐行原地更新即可，空间从 O(mn) 降到 O(n)。
- **障碍处理**：障碍格的路径数直接置 0，相当于"这条路到此断掉"，后续格子自然加不到它的贡献。
- **逆向需求状态（Dungeon Game）**：正向"累计到达某格时的血量/最小花费"在这里行不通，因为一路上血量的最低点才是瓶颈，加血晚到救不了前面已死的人。所以把状态定义反过来：`dp[r][c]` 表示"踏进 (r,c) 这格之前必须至少有多少血，才能保证从这格出发走到终点全程不死"。这种"需要多少"的状态只能从终点往起点倒推。
- **无穷大哨兵 `float('inf')`**：在 min 转移里代表"这条路不存在/还没算出来"，任何有限数和它比 min 都会赢，从而自动屏蔽无效来源。

## 3. 思路推导：从小例子开始

**问题 1：数路径（`unique_paths`）**

- Step 1：第一行的每个格子都只有 1 条路（一路向右），所以初始化 `dp=[1]*n`。
- Step 2：对下面的每一行，格子 (r,c) 的路径数 = (r−1,c) 的路径数 + (r,c−1) 的路径数。用一维数组时，`dp[c]` 在更新前恰好存着上一行的值（上方来的），`dp[c-1]` 已被更新为本行的值（左方来的），所以 `dp[c]+=dp[c-1]` 一句就完成了合并。
- Step 3：滚动 m−1 行后，`dp[-1]` 就是右下角的路径数。

手算 `unique_paths(3,3)`：初始 `[1,1,1]`；滚完第 2 行得 `[1,2,3]`；滚完第 3 行得 `[1,3,6]`。答案 6，你可以枚举验证：RRDD、RDRD、RDDR、DRRD、DRDR、DDRR 正好 6 条。

**问题 2：带障碍数路径（`unique_paths_with_obstacles`）**

- Step 1：与上面同理，只是多一条：格子是障碍（x 为真）就把 `dp[c]` 清零。
- Step 2：起点若是障碍，第一次循环 `dp[0]` 就被清零，整张表全零，答案自然是 0。

手算 `[[0,0],[0,1]]`：初始 `dp=[1,0]`；第一行 `[0,0]`：c=0 空地保持 1，c=1 空地 `dp[1]+=dp[0]` 得 1，此时 `dp=[1,1]`；第二行 `[0,1]`：c=0 空地保持 1，c=1 是障碍清零，`dp=[1,0]`，返回 0。

**问题 3：最小路径和（`min_path_sum`）**

- Step 1：`dp[c]` 表示"走到本行 c 列的最小总花费"，来自上方（旧 `dp[c]`）或左方（新 `dp[c-1]`）中较便宜的一边，再加上本格花费 x。
- Step 2：每行开头把 `dp[0]=0`、其余设为无穷大，这样第一行第一格的花费会自然算成自身值，而"从左边进入第 0 列"这种不存在的来源被无穷大挡掉。

手算 `[[1,3,1],[1,5,1]]`：初始 `dp=[0,inf,inf]`；第一行算完 `dp=[1,4,5]`；第二行：c=0 得 1+1=2，c=1 得 min(4,2)+5=7，c=2 得 min(5,7)+1=6。答案 6。

**问题 4：地牢（`calculate_minimum_hp`，逆向推）**

- Step 1：定义 `dp[r][c]` = 踏进 (r,c) 之前至少要有的血量。要求只有两条：任何时刻血量 ≥ 1；踏进格子后血量变化（扣血或加血），剩余血量必须够应付"下一步出发所需"。
- Step 2：从终点右下角开始：踏进终点前至少要有 1 点血，这是底线。
- Step 3：一般格子 (r,c)：先看右、下两个后继里"更省血的那个"（取 min），设为 `after`；本格血量变化是 `dungeon[r][c]`，那么进入本格前的血量 h 要满足 h + dungeon[r][c] >= after，即 h >= after − dungeon[r][c]；同时 h 本身至少是 1。所以 `dp[r][c] = max(1, after − dungeon[r][c])`。
- Step 4：一路倒推到左上角，`dp[0][0]` 就是出发时的最低初始血量。

“运行示例”部分 用地牢 `[[-2,-3,3],[-5,-10,1],[10,30,-5]]` 演示了整张倒推表（每行是"进入该行各格所需的最少血量"）：

| 行 | 进入各格最少血量 |
|---|---|
| 0 | 7, 5, 2 |
| 1 | 6, 11, 5 |
| 2 | 1, 1, 6 |

我们从右下角逆推验证几个格子：终点 (2,2) 是 −5，进入前需 max(1, 1−(−5))=6；(2,1) 是 +30 加血格，进入前只需 max(1, 6−30)=1；(1,1) 是 −10 大扣血格，后继只需 min(1,5)=1，所以需 max(1, 1+10)=11；最后起点 (0,0) 是 −2，后继需 min(6,5)=5，所以需 max(1, 5+2)=7。最终答案就是 7。

## 4. 核心代码：unique_paths

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def unique_paths(m, n):
    if m <= 0 or n <= 0:
        return 0
    dp = [1] * n
    for _ in range(1, m):
        for c in range(1, n):
            dp[c] += dp[c - 1]
    return dp[-1]
```

### 逐段读懂代码

**函数一：`unique_paths`**

```python
def unique_paths(m, n):
    if m <= 0 or n <= 0:
        return 0
    dp = [1] * n
    for _ in range(1, m):
        for c in range(1, n):
            dp[c] += dp[c - 1]
    return dp[-1]
```

- `if m<=0 or n<=0: return 0`：挡住非正数的行数/列数，这种网格不存在路径，返回 0。
- `dp=[1]*n`：第一行每个格子都只有"一路向右"这 1 条路。
- `for _ in range(1,m)`：滚完剩余 m−1 行，下划线表示我们不需要行号。
- `dp[c]+=dp[c-1]`：更新前的 `dp[c]` 是上方格子的路径数，更新后的 `dp[c-1]` 是左方格子的路径数，一句加法完成转移。
- `return dp[-1]`：数组最后一个元素就是右下角。

**函数二：`unique_paths_with_obstacles`**

```python
def unique_paths_with_obstacles(grid):
    if not grid or not grid[0]:
        return 0
    dp = [0] * len(grid[0])
    dp[0] = 1
    for row in grid:
        for c, x in enumerate(row):
            if x:
                dp[c] = 0
            elif c:
                dp[c] += dp[c - 1]
    return dp[-1]
```

- `if not grid or not grid[0]`：空网格或空行直接返回 0（教学契约里显式定义了这个边界）。
- `dp=[0]*...; dp[0]=1`：先给起点记 1 条"虚拟路"，扫第一行时会把它正确传递下去。
- `if x: dp[c]=0`：x 非零即障碍，本格路径数清零，后面格子加不到它。
- `elif c: dp[c]+=dp[c-1]`：c=0（第一列）且非障碍时什么都不做，`dp[0]` 原样保留——它继承"上方格子是否可达"，一旦上方有障碍它早被清零，会一路零下去，这正是想要的效果。

**函数三：`min_path_sum`**

```python
def min_path_sum(grid):
    if not grid or not grid[0]:
        return 0
    dp = [float('inf')] * len(grid[0])
    dp[0] = 0
    for row in grid:
        for c, x in enumerate(row):
            dp[c] = min(dp[c], dp[c - 1] if c else float('inf')) + x
    return dp[-1]
```

- `dp[0]=0` 配合其余为无穷大：让第一格的花费算出来等于自身（min(0, inf)+x = x），同时挡掉不存在的"左来源"。
- `dp[c-1] if c else float('inf')`：这是三元表达式的紧凑写法——c 大于 0 时提供左格子的值，c 等于 0 时用无穷大表示"没有左邻居"。
- 整句含义：本格最小花费 = min(上方来的累计, 左方来的累计) + 本格值。

**函数四：`calculate_minimum_hp`**

```python
def calculate_minimum_hp(dungeon):
    if not dungeon or not dungeon[0]:
        return 1
    m, n = (len(dungeon), len(dungeon[0]))
    dp = [float('inf')] * (n + 1)
    dp[n - 1] = 1
    for r in range(m - 1, -1, -1):
        for c in range(n - 1, -1, -1):
            dp[c] = max(1, min(dp[c], dp[c + 1]) - dungeon[r][c])
    return dp[0]
```

- `if not ...: return 1`：空地牢也定义成"至少 1 点血"。
- `dp=[float('inf')]*(n+1); dp[n-1]=1`：开 n+1 个位置，多出的末尾哨兵 `dp[n]` 永远是无穷大，代表网格右边界外"不存在"。`dp[n-1]=1` 是终点的底线：进入终点前至少 1 点血。
- `for r in range(m-1,-1,-1)` 与内层同理：下标从大到小，实现"从右下角往左上角"的倒序遍历。
- `min(dp[c],dp[c+1])`：滚动数组里，更新前的 `dp[c]` 是"下一行同列"（下方格子）的需求，`dp[c+1]` 是"本行右邻"的需求（本行从右往左扫，右邻先算好了），取较小者作为后继需求。
- `max(1,...-dungeon[r][c])`：先减去本格血量变化得到进入前需求，再用 max(1, ...) 挡住"加血太多导致算出 0 或负数"的情况——血量再富裕，进入任何格子前也至少得有 1 点。
- `return dp[0]`：倒推到左上角，就是出发前所需的最少血量。

## 5. 分段实现：按顺序运行

**本章接口：** `unique_paths(m, n)`、`unique_paths_with_obstacles(grid)`、`min_path_sum(grid)`、`calculate_minimum_hp(dungeon)`

### unique_paths

In [1]:
def unique_paths(m, n):
    if m <= 0 or n <= 0:
        return 0
    dp = [1] * n
    for _ in range(1, m):
        for c in range(1, n):
            dp[c] += dp[c - 1]
    return dp[-1]

### unique_paths_with_obstacles

In [2]:
def unique_paths_with_obstacles(grid):
    if not grid or not grid[0]:
        return 0
    dp = [0] * len(grid[0])
    dp[0] = 1
    for row in grid:
        for c, x in enumerate(row):
            if x:
                dp[c] = 0
            elif c:
                dp[c] += dp[c - 1]
    return dp[-1]

### min_path_sum

In [3]:
def min_path_sum(grid):
    if not grid or not grid[0]:
        return 0
    dp = [float('inf')] * len(grid[0])
    dp[0] = 0
    for row in grid:
        for c, x in enumerate(row):
            dp[c] = min(dp[c], dp[c - 1] if c else float('inf')) + x
    return dp[-1]

### calculate_minimum_hp

In [4]:
def calculate_minimum_hp(dungeon):
    if not dungeon or not dungeon[0]:
        return 1
    m, n = (len(dungeon), len(dungeon[0]))
    dp = [float('inf')] * (n + 1)
    dp[n - 1] = 1
    for r in range(m - 1, -1, -1):
        for c in range(n - 1, -1, -1):
            dp[c] = max(1, min(dp[c], dp[c + 1]) - dungeon[r][c])
    return dp[0]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

grid=[[-2,-3,3],[-5,-10,1],[10,30,-5]]; dp=[[0]*3 for _ in range(3)]
for r in range(2,-1,-1):
    for c in range(2,-1,-1):
        after=1 if (r,c)==(2,2) else min([dp[nr][nc] for nr,nc in [(r+1,c),(r,c+1)] if nr<3 and nc<3])
        dp[r][c]=max(1,after-grid[r][c])
show_table(['行','进入各格最少血量'],list(enumerate(dp)))

行,进入各格最少血量
0,"[7, 5, 2]"
1,"[6, 11, 5]"
2,"[1, 1, 6]"


## 7. 正确性、适用条件与复杂度

每条路径的最后或下一步只有两个互斥方向，DAG顺序确保依赖已计算。地牢从终点向前归纳：进入血量h满足h≥1且h+格值≥下一格所需，故h=max(1,next−格值)。

**代价：** O(mn)算术操作、O(n)辅助空间。矩形输入；教学扩展的空网格结果已显式定义。路径计数是任意精度整数，不隐式取模。

### 展开理解

**为什么正向推前三个问题是对的**：任何一个到达格子 (r,c) 的走法，最后一步要么从上面下来，要么从左边过来，只有这两种可能，而且两种走法互不相同、也覆盖了全部走法。所以数路径就把两类的数量**相加**，求最小花费就把两类的最小值**取 min** 再加本格。又因为"只能右、下"保证了格子之间的依赖不会成环，我们按行从上到下、行内从左到右计算时，上方和左方的值一定已经算好，一路推到右下角就是正确答案。

**为什么地牢必须倒着推**：如果像最小路径和那样正着累计"到达本格时的血量"，会碰上一个致命问题——你不知道该记录"当前血量"还是"一路上的最低血量"，因为后面遇到的扣血量会改变对前面的要求（后面扣得狠，前面就得带更多血出发）。把状态换成"从本格出发活着走到终点，进入前至少要多少血"，这个量只取决于本格和后继格子，与更前面的历史完全无关，所以从终点出发倒着算，每格的需求都能由右、下两个后继直接确定。正确性论证是：进入 (r,c) 前的血量 h 必须同时满足两个条件——h 至少是 1（活着），以及 h 加上本格变化后不少于后继所需，即 h ≥ 后继需求 − 本格值。两个下限同时满足就是取较大者，这正是代码里 `max(1, min(dp[c],dp[c+1])-dungeon[r][c])` 的含义。

**复杂度**：四个函数都把每个格子看一遍、每格只做常数次加减和比较，所以时间是 O(mn)。比如 100×100 的网格就是 1 万格，瞬间算完；即便 1000×1000（100 万格）也只是百万级操作，一秒内完事。空间上全部用一维数组滚动，只需要 O(n)；地牢那题因为要看右邻，数组多开了一个哨兵位，仍然是 O(n)。

## 8. 单元测试：每个用例在检查什么

```python
assert unique_paths(3, 7) == 28
```

- 正常值测试：3 行 7 列的路径数是组合数 C(8,2)=28，验证转移加法没写错。

```python
assert unique_paths_with_obstacles([[1]]) == 0

assert unique_paths_with_obstacles([[0, 0], [0, 1]]) == 0
```

- 第一条是"起点即障碍"的极端用例：1×1 网格且唯一格是障碍，答案必须为 0，检验 `dp[0]=1` 的初始路会不会被错误保留。
- 第二条是"终点被堵"用例：只有终点是障碍，所有路都断了，检验障碍清零逻辑。

```python
assert calculate_minimum_hp([[-5]]) == 6
```

- 单格扣 5 血的边界用例：进入前需要 6 点，扣完剩 1 恰好满足"至少 1"。它专门检验初始化 `dp[n-1]=1` 和 `max(1,...)` 这两处底线逻辑。

```python
assert calculate_minimum_hp([[-2, -3, 3], [-5, -10, 1], [10, 30, -5]]) == 7
```

- 正常值测试：LeetCode 174 的经典样例，答案 7，可以对照第三节的倒推表逐格核对。

```python
for a in product((-2, 1, 3), repeat=6):
    g = [a[:3], a[3:]]
    p = paths(g)
    assert min_path_sum(g) == min(map(sum, p))
    need = min((max(1, 1 - min((sum(x[:k]) for k in range(1, len(x) + 1)))) for x in p))
    assert calculate_minimum_hp(g) == need
```

- 这是暴力对拍：枚举 3⁶=729 个 2×3 网格，`paths` 递归列出每条具体路径。`min_path_sum` 与"所有路径和的最小值"对照；地牢需求则对每条路径逐位置算前缀和、找出全程最低点，推出该路径所需的初始血量，再取所有路径中最省的，与 DP 结果对照。穷举法与递推法在几百个输入上完全一致，是正确性的强力证据。

In [6]:
assert unique_paths(3, 7) == 28

assert unique_paths_with_obstacles([[1]]) == 0

assert unique_paths_with_obstacles([[0, 0], [0, 1]]) == 0

assert calculate_minimum_hp([[-5]]) == 6

assert calculate_minimum_hp([[-2, -3, 3], [-5, -10, 1], [10, 30, -5]]) == 7

from itertools import product

def paths(g, r=0, c=0):
    if r == len(g) - 1 and c == len(g[0]) - 1:
        return [[g[r][c]]]
    return [[g[r][c]] + tail for nr, nc in [(r + 1, c), (r, c + 1)] if nr < len(g) and nc < len(g[0]) for tail in paths(g, nr, nc)]

for a in product((-2, 1, 3), repeat=6):
    g = [a[:3], a[3:]]
    p = paths(g)
    assert min_path_sum(g) == min(map(sum, p))
    need = min((max(1, 1 - min((sum(x[:k]) for k in range(1, len(x) + 1)))) for x in p))
    assert calculate_minimum_hp(g) == need

print('N100: 所有本章断言通过')

N100: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 对比最小路径和与最低初始生命值的状态。

**练习 2：** 滚动到一行。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（对比最小路径和与最低初始血量）**：提示——试着把 `min_path_sum` 的状态定义（"到达本格为止的累计花费"）直接搬到地牢题上，找一个它失败的小例子（比如先扣血后大量加血的网格），想清楚它为什么错：累计量是"过去决定现在"，而血量要求是"未来约束现在"。写清楚两个状态分别依赖哪个方向（正推 vs 倒推）、合并符号为什么一个是 min 一个是 max，再拿 `[[1,-3,3],[1,1,1]]` 这类小网格手算对比。
- **练习 2（滚动到一行）**：提示——本章代码其实已经滚动到一行了，你要做的是把"为什么 `dp[c]` 更新前存的是上一行、更新后变成本行"这件事讲清楚并画出更新前后的对照表。可以拿 3×3 网格把每行滚动前后的数组值手抄下来，验证原地更新不丢信息；再想一想如果转移还需要"左上方"的格子（某些变形题），纯原地更新会不会被覆盖，该怎么调整遍历顺序或额外存一个变量。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
def unique_paths(m, n):
    if m <= 0 or n <= 0:
        return 0
    dp = [1] * n
    for _ in range(1, m):
        for c in range(1, n):
            dp[c] += dp[c - 1]
    return dp[-1]

def unique_paths_with_obstacles(grid):
    if not grid or not grid[0]:
        return 0
    dp = [0] * len(grid[0])
    dp[0] = 1
    for row in grid:
        for c, x in enumerate(row):
            if x:
                dp[c] = 0
            elif c:
                dp[c] += dp[c - 1]
    return dp[-1]

def min_path_sum(grid):
    if not grid or not grid[0]:
        return 0
    dp = [float('inf')] * len(grid[0])
    dp[0] = 0
    for row in grid:
        for c, x in enumerate(row):
            dp[c] = min(dp[c], dp[c - 1] if c else float('inf')) + x
    return dp[-1]

def calculate_minimum_hp(dungeon):
    if not dungeon or not dungeon[0]:
        return 1
    m, n = (len(dungeon), len(dungeon[0]))
    dp = [float('inf')] * (n + 1)
    dp[n - 1] = 1
    for r in range(m - 1, -1, -1):
        for c in range(n - 1, -1, -1):
            dp[c] = max(1, min(dp[c], dp[c + 1]) - dungeon[r][c])
    return dp[0]

# 示例与回归测试
assert unique_paths(3, 7) == 28

assert unique_paths_with_obstacles([[1]]) == 0

assert unique_paths_with_obstacles([[0, 0], [0, 1]]) == 0

assert calculate_minimum_hp([[-5]]) == 6

assert calculate_minimum_hp([[-2, -3, 3], [-5, -10, 1], [10, 30, -5]]) == 7

from itertools import product

def paths(g, r=0, c=0):
    if r == len(g) - 1 and c == len(g[0]) - 1:
        return [[g[r][c]]]
    return [[g[r][c]] + tail for nr, nc in [(r + 1, c), (r, c + 1)] if nr < len(g) and nc < len(g[0]) for tail in paths(g, nr, nc)]

for a in product((-2, 1, 3), repeat=6):
    g = [a[:3], a[3:]]
    p = paths(g)
    assert min_path_sum(g) == min(map(sum, p))
    need = min((max(1, 1 - min((sum(x[:k]) for k in range(1, len(x) + 1)))) for x in p))
    assert calculate_minimum_hp(g) == need

print('N100: 所有本章断言通过')

N100: 所有本章断言通过


## 11. 代表题与迁移

- **62. Unique Paths（不同路径）**：练"只能右下走"的基本路径计数转移，对应 `unique_paths`。
- **63. Unique Paths II（不同路径 II）**：在 62 的基础上加障碍清零，对应 `unique_paths_with_obstacles`。
- **64. Minimum Path Sum（最小路径和）**：把"计数相加"换成"最值取 min"，对应 `min_path_sum`。
- **174. Dungeon Game（地下城游戏）**：练本章的核心思维转弯——状态必须定义成"需求"并从终点倒推，对应 `calculate_minimum_hp`。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。